- https://simonxin.com/blogs/llm_reasoning_materials/index.html

In [1]:
from IPython.display import Image

In [2]:
Image(url='./figs/grpo_highlights.png', width=400)

In [7]:
Image(url='https://simonxin.com/blogs/llm_reasoning_materials/grpo-1.png', width=500)

In [6]:
Image(url='https://simonxin.com/blogs/llm_reasoning_materials/grpo-2.png', width=500)

- $\pi_\theta(y_{i,t}|x,y_{i,<t})$ is the **token-wise likelihood** on one training sample $(x,y)$
    - This term is identical to the pre-training loss. The difference is that GRPO involves two additional scaling factors:
        - $\frac{1}{\pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}$
        - $\hat{A}_{i,t}$

### token-wise issue??

- $(x, y_i)$ is sampled from $\pi_{\theta_\text{old}}$ but used to update $\pi_\theta$, However, the possible value of $\frac{1}{\pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}$ ranges from  1 to $\infty$
- https://arxiv.org/pdf/2505.12929
    - Do Not Let **Low-Probability Tokens** Over-Dominate in RL for LLMs
    - $C_l \cdot |w_{i,t}| \cdot \sqrt{\frac{N}{N-1}} \cdot (1 - \pi_\theta(o_{i,t})) \le ||\delta_l(o_{i,t})|| \le D_l \cdot |w_{i,t}| \cdot \sqrt{2} \cdot (1 - \pi_\theta(o_{i,t}))$
        - 梯度的范数（即梯度的大小）||δl(oi,t)|| 与 (1 - πθ(oi,t)) 这一项成正比。
    - 在计算上，低概率词元可以被看作是那些在强化学习更新步骤中，贡献了绝大部分梯度信号的“离群点（outliers）”。
        - 由于低概率词元的梯度值比高概率词元大几个数量级，当它们被一起平均时，低概率词元的梯度信号会“淹没”高概率词元的梯度信号。

## (sequence) length bias in RL?

> Implementation + data > algorithm

Three different ways to sum loss. (Setup: G is a group of completions per a question, each response has o tokens.)

In [3]:
Image(url='./figs/length_bias.png', width=400)

### GSPO

- verl/trainer/config/actor/actor.yaml
    - policy_loss.loss_mode: vanilla / clip-cov / kl-cov / gpg / gspo

$$
\mathcal{J}_{\text{GRPO}} = \frac{1}{|y_i|}\sum_{t=1}^{|y_i|} \frac{\pi_\theta(y_{i,t}|x,y_{i,<t})}{\pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}
$$
$$
\begin{aligned}     \mathcal{J}_{\text{GSPO}} &= \left(\frac{\pi_\theta(y_i|x)}{\pi_{\theta_\text{old}}(y_i|x)}\right)^{\frac{1}{|y_i|}} \\     & = \left(\frac{\Pi_{t=1}^{|y_i|} \pi_\theta(y_{i,t}|x,y_{i,<t})}{\Pi_{t=1}^{|y_i|} \pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}\right)^{\frac{1}{|y_i|}} \\     & = \exp\left(\frac{1}{|y_i|}\log\left(     \frac{\Pi_{t=1}^{|y_i|} \pi_\theta(y_{i,t}|x,y_{i,<t})}{\Pi_{t=1}^{|y_i|} \pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}     \right)\right) \\     & = \exp\left(\frac{1}{|y_i|} \sum_{t=1}^{|y_i|} \log\left(     \frac{ \pi_\theta(y_{i,t}|x,y_{i,<t})}{ \pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}     \right)\right)     \end{aligned}
$$

- GRPO is doing the arithmetic mean of $\frac{\pi_\theta(y_{i,t}|x,y_{i,<t})}{\pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}$
- GSPO is doing the geometric mean of $\frac{\pi_\theta(y_{i,t}|x,y_{i,<t})}{\pi_{\theta_\text{old}}(y_{i,t}|x,y_{i,<t})}$
    - $\sqrt[n]{a_1 a_2 \cdots a_n \vphantom{t}} = \exp \left( \frac{\ln a_1 + \ln a_2 + \cdots + \ln a_n }{n} \right)$ ($x=\exp(\log x)$)

In [11]:
Image(url='https://simonxin.com/blogs/llm_reasoning_materials/gspo_loss.png', width=600)

In [4]:
Image(url='https://simonxin.com/blogs/llm_reasoning_materials/grpo_gradient.png', width=500)

- In GSPO, all tokens in a sequence share the same scaling factor $\left(\frac{\pi_\theta(y_i|x)}{\pi_{\theta_\text{old}}(y_i|x)}\right)^{\frac{1}{|y_i|}}$
    - $s_i(\theta)$
- 损失的**数值大小（magnitude）由  $s_i(\theta)$ 决定，但梯度的方向（direction）仍然由单个词元的策略梯度 $\nabla_\theta\log\pi_\theta(y_{i,t}|x,y_{i,\lt t})$
    - 用整个团队（序列）的整体表现来决定奖励的“大小”，但根据每个队员（词元）自己的行为来决定奖励的“方向”。

#### verl impl (core-algos.py)

- `log_seq_importance_ratio = log_prob - log_prob.detach() + negative_approx_kl_seq.detach().unsqueeze(-1)`
    - $s_{i,t}(θ) = \text{sg}[s_i(θ)] \frac{ π_θ(y_{i,t}|x, y_{i,<t})}{\text{sg}[π_θ(y_{i,t}|x, y_{i,<t})]}$
    - $\log(s_{i,t}(θ)) = \text{sg}[\log(s_i(θ))] + \text{log\_prob} - \text{sg}[\text{log\_prob}]$
        - 变化主要是为了保留 $\text{log\_prob} = π_θ(y_{i,t}|x, y_{i,<t})$ 这部分 backward 时的梯度
    - $\exp(\log(s_{i,t}(θ)))$ 数值上等等于 $s_i(\theta)$
    - `negative_approx_kl_seq.detach()`: 负责提供更新的**“大小”或“幅度”**。它代表了那个被你写在纸上、固定不变的团队总分。因为 .detach()，它的数值会被使用，但它的计算过程不会影响到任何一个词元的梯度方向。
    - `log_prob - log_prob.detach()`: 负责提供更新的**“方向”**。这是一个编程技巧，它的数值是 0（不影响奖励大小），但在计算梯度时，只有 log_prob 这一项会生效。这保证了算法知道，这份奖励应该用在“提升当前词元概率”这个方向上。
- 前向传播 (Forward Pass): log_prob - log_prob.detach() 的值为 0，所以 log_seq_importance_ratio 的值就是 negative_approx_kl_seq，即 $\log s_i(\theta)$，因此，计算出的损失值确实是基于 $s_i(\theta)$ 的
- stop-gradient
    - 在 pytorch 中通过 detach 实现

$$
s_i(\theta) = \left( \frac{\pi_{\theta}(y_i|x)}{\pi_{\theta_{old}}(y_i|x)} \right)^{\frac{1}{|y_i|}} = \exp\left[ \frac{1}{|y_i|} \sum_t \log\left( \frac{\pi_{\theta}(y_{i,t}|x, y_{i,<t})}{\pi_{\theta_{old}}(y_{i,t}|x, y_{i,<t})} \right) \right]
$$

$$
s_{i,t}(\theta) = \text{sg}[s_i(\theta)] \cdot \frac{\pi_{\theta}(y_{i,t}|x, y_{i,<t})}{\text{sg}[\pi_{\theta}(y_{i,t}|x, y_{i,<t})]}
$$

$$
\log(s_{i,t}(\theta)) = \text{sg}[\log(s_i(\theta))] + \text{log\_prob} - \text{sg}[\text{log\_prob}]
$$

- GSPO 的核心思想是，对于一个序列 y_i 中的每一个词元 y_{i,t}，它的策略梯度更新都应该被同一个序列级别的因子 s_i(θ) 来缩放。
- 直接在代码中实现上述 J_GSPO(θ) 目标会遇到一个问题：损失函数中的 s_i(θ) 依赖于整个序列的对数概率 log_prob。当自动微分框架（如 PyTorch）计算梯度时，对于序列中的某个特定词元 y_{i,t}，它的梯度会受到序列中所有其他词元 y_{i,j} (j≠t) 的影响，因为它们都出现在 s_i(θ) 的计算中。
    - 如果用这个 $s_{i,t}(θ)$ 来构造一个 PPO 风格的词元级别损失 $L_t = s_{i,t}(\theta) \cdot \hat{A}_i$，它的梯度 ∇_θ L_t 是什么？
        - $∇_θ L_t = (∇_θ s_{i,t}(\theta)) \cdot \hat{A}_i$
        - $∇_θ s_{i,t}(\theta)$。由于 `sg[...]` 部分被视为常数，我们可以把它们记作 $C_1 = sg[s_i(θ)]$ 和 $C_2 = sg[π_θ(y_{i,t}|...)]$。
            - $s_{i,t}(\theta) = \frac{C_1}{C_2} \cdot \pi_{\theta}(y_{i,t}|x, y_{i,<t})$
        - 对上式求导： $∇_θ s_{i,t}(\theta) = \frac{C_1}{C_2} \cdot \nabla_θ \pi_{\theta}(y_{i,t}|x, y_{i,<t})$

```python
@register_policy_loss("gspo")
def compute_policy_loss_gspo(
    old_log_prob: torch.Tensor,
    log_prob: torch.Tensor,
    advantages: torch.Tensor,
    response_mask: torch.Tensor,
    loss_agg_mode: str = "seq-mean-token-mean",
    config: Optional[DictConfig | ActorConfig] = None,
    rollout_log_probs: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Compute the clipped policy objective and related metrics for GSPO.

    See https://arxiv.org/pdf/2507.18071 for more details.

    Args:
        old_log_prob (torch.Tensor):
            Log-probabilities of actions under the old policy, shape (batch_size, response_length).
        log_prob (torch.Tensor):
            Log-probabilities of actions under the current policy, shape (batch_size, response_length).
        advantages (torch.Tensor):
            Advantage estimates for each action, shape (batch_size, response_length).
        response_mask (torch.Tensor):
            Mask indicating which tokens to include in the loss, shape (batch_size, response_length).
        loss_agg_mode (str, optional):
            Aggregation mode for `agg_loss`. For GSPO, it is recommended to use "seq-mean-token-mean".
    """

    assert config is not None
    assert isinstance(config, ActorConfig)
    clip_ratio_low = config.clip_ratio_low if config.clip_ratio_low is not None else config.clip_ratio
    clip_ratio_high = config.clip_ratio_high if config.clip_ratio_high is not None else config.clip_ratio

    negative_approx_kl = log_prob - old_log_prob

    # compute sequence-level importance ratio:
    # si(θ) = (π_θ(yi|x)/π_θold(yi|x))^(1/|yi|) =
    # exp [(1/|y_i|) * Σ_t log(π_θ(y_i,t|x,y_i,<t)/π_θold(y_i,t|x,y_i,<t))]
    seq_lengths = torch.sum(response_mask, dim=-1).clamp(min=1)
    negative_approx_kl_seq = torch.sum(negative_approx_kl * response_mask, dim=-1) / seq_lengths

    # Combined ratio at token level:
    # s_i,t(θ) = sg[s_i(θ)] · π_θ(y_i,t|x, y_i,<t) / sg[π_θ(y_i,t|x, y_i,<t)]
    # In log space: log(s_i,t(θ)) = sg[log(s_i(θ))] + log_prob - sg[log_prob]
    log_seq_importance_ratio = log_prob - log_prob.detach() + negative_approx_kl_seq.detach().unsqueeze(-1)
    log_seq_importance_ratio = torch.clamp(log_seq_importance_ratio, max=10.0)  # clamp for numerical stability

    # finaly exp() to remove log
    seq_importance_ratio = torch.exp(log_seq_importance_ratio)

    pg_losses1 = -advantages * seq_importance_ratio
    pg_losses2 = -advantages * torch.clamp(seq_importance_ratio, 1 - clip_ratio_low, 1 + clip_ratio_high)
    pg_losses = torch.maximum(pg_losses1, pg_losses2)

    # for GSPO, we need to aggregate the loss at the sequence level (seq-mean-token-mean)
    pg_loss = agg_loss(loss_mat=pg_losses, loss_mask=response_mask, loss_agg_mode="seq-mean-token-mean")

    # For compatibility, return zero for pg_clipfrac_lower (not used in standard GSPO)
    pg_clipfrac = verl_F.masked_mean(torch.gt(pg_losses2, pg_losses1).float(), response_mask)
    pg_clipfrac_lower = torch.tensor(0.0, device=pg_loss.device)

    ppo_kl = verl_F.masked_mean(-negative_approx_kl, response_mask)

    return pg_loss, pg_clipfrac, ppo_kl, pg_clipfrac_lower
```

### cases

https://chatgpt.com/c/68d24a2c-bb58-8332-968c-0eb215bff56a
- vanilla grpo: $\frac{-A_{i,t}r_{i,t}}{N}\cdot m_{i,t}$
- gspo gradient: $\frac{-A_{i,t}s_i}{GL_i}$

In [22]:
import torch
torch.manual_seed(42)

B, T = 2, 4
log_prob = torch.tensor([[-2.1,-1.7,-1.3,-2.9],
                         [-1.4,-1.0,-2.4,-1.8]], dtype=torch.float32, requires_grad=True)
old_log_prob = torch.tensor([[-2.2,-1.5,-0.9,-3.0],
                             [-1.0,-1.2,-2.0,-2.5]], dtype=torch.float32)

advantages = torch.tensor([[ 0.8,-0.3, 0.5,0.1],
                           [-0.6, 0.4,-0.2,0.7]], dtype=torch.float32)

mask = torch.tensor([[1.,1.,1.,0.],
                     [1.,1.,1.,1.]], dtype=torch.float32)

eps_low=eps_high=0.20

In [6]:
def token_mean(x,m): return (x*m).sum() / m.sum().clamp(min=1.)
def seq_mean_token_mean(x,m):
    L = m.sum(-1).clamp(min=1.)
    return ((x*m).sum(-1)/L).mean()

In [7]:
# ---------- Vanilla (PPO/GRPO-style) ----------
log_ratio = torch.clamp(log_prob - old_log_prob, -20, 20)
ratio = torch.exp(log_ratio)
pg1 = -advantages * ratio
pg2 = -advantages * torch.clamp(ratio, 1-eps_low, 1+eps_high)
loss_van = token_mean(torch.maximum(pg1, pg2), mask)
log_prob.grad=None
loss_van.backward(retain_graph=True)
van_grad = log_prob.grad.clone().detach()

In [8]:
print("Vanilla grad dL/d(log_prob):\n", van_grad)

Vanilla grad dL/d(log_prob):
 tensor([[-0.1263,  0.0351, -0.0479, -0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000]])


In [21]:
advantages[0], ratio[0]

(tensor([ 0.8000, -0.3000,  0.5000,  0.1000]),
 tensor([1.1052, 0.8187, 0.6703, 1.1052], grad_fn=<SelectBackward0>))

In [17]:
pg1[0] / 7

tensor([-0.1263,  0.0351, -0.0479, -0.0158], grad_fn=<DivBackward0>)

In [9]:
# ---------- GSPO ----------
seq_len = mask.sum(-1).clamp(min=1.)
barDelta = ((log_prob-old_log_prob)*mask).sum(-1)/seq_len
log_s = barDelta.detach().unsqueeze(-1) + (log_prob - log_prob.detach())
log_s = torch.clamp(log_s, max=10.)
s = torch.exp(log_s)

gspo_unclipped = -advantages * s
gspo_clipped   = -advantages * torch.clamp(s, 1-eps_low, 1+eps_high)
loss_gspo = seq_mean_token_mean(torch.maximum(gspo_unclipped, gspo_clipped), mask)
log_prob.grad=None
loss_gspo.backward()
gspo_grad = log_prob.grad.clone().detach()

In [11]:
s

tensor([[0.8465, 0.8465, 0.8465, 0.8465],
        [1.0253, 1.0253, 1.0253, 1.0253]], grad_fn=<ExpBackward0>)

In [10]:
print("GSPO    grad dL/d(log_prob):\n", gspo_grad)

GSPO    grad dL/d(log_prob):
 tensor([[-0.1129,  0.0423, -0.0705, -0.0000],
        [ 0.0769, -0.0513,  0.0256, -0.0897]])


In [13]:
- advantages[1] * 1.0253 / 8

tensor([ 0.0769, -0.0513,  0.0256, -0.0897])